<a href="https://colab.research.google.com/github/atenine/LING-230-Single-Model-Disfluency-Detector/blob/optimization/scripts/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from huggingface_hub import login
from dotenv import dotenv_values
import pandas as pd
from transformers import AutoConfig, AutoModel
import torchaudio
import torch
import torch.nn as nn
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler

%load_ext tensorboard

login(dotenv_values()['HUGGINGFACE_API_KEY'])



In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install gdown

import os
import tarfile

# Ensure /content/data directory exists for extraction
os.makedirs('/content/data', exist_ok=True)

# Google Drive file ID for the zip file provided by the user
file_id = '18I3VYO_Le-_cFY6Rmbg1MLbHrwmwPE6F'
zip_file_name = '/content/data/SEP-28k_CLIP.zip'

print(f"Downloading {zip_file_name} from Google Drive...")
# Download the file from Google Drive
!gdown --id "$file_id" -O "$zip_file_name"

print(f"Extracting {zip_file_name}...")
# Extract the zip file to /content/data/
with tarfile.open(zip_file_name, 'r') as tar:
    tar.extractall('/content/data/')

# Define the paths based on the extracted data
clips_path = '/content/data/SEP-28k_CLIP'


print("Data setup complete.")
print(f"Clips path set to: {clips_path}")
print("Clips directory exists:", os.path.exists(clips_path))

In [ ]:
csv_path   = '/content/SEP-28k-Extended_clips.csv'
clips_path = '/content/data/SEP-28k_CLIP'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

In [ ]:
# Check CSV
print("CSV exists:", os.path.exists(csv_path))

# Check one audio clip
path = clips_path + '/HeStutters/10/HeStutters_10_0.wav'
print("Data Exists:", os.path.exists(path))


In [ ]:
# Load CSV
df = pd.read_csv(csv_path)

# Build filepath — Show/EpId/Show_EpId_ClipId.wav
df['filepath'] = df.apply(
    lambda row: os.path.join(
        clips_path,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)

# Check how many files exist
df['exists'] = df['filepath'].apply(os.path.exists)
print(f"Total clips in CSV:     {len(df)}")
print(f"Clips found on disk:    {df['exists'].sum()}")
print(f"Clips NOT found:        {(~df['exists']).sum()}")

# Keep only existing files
df = df[df['exists']]
print("\nSample filepath:")
print(df['filepath'].iloc[0])

In [ ]:
stutter_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']
label_map={
    'Prolongation': 'prolongation',
    'Block': 'block',
    'SoundRep': 'soundrep',
    'WordRep': 'wordrep',
    'Interjection': 'interjection',
    'NoStutteredWords':'fluent'
}
cols_to_normalize = list(label_map.keys())
df[list(label_map.values())] = df[cols_to_normalize].div(3.0).rename(columns=label_map)

print("\nAverage probability per stutter type:")
for new_col in list(label_map.values())[:-1]:
    print(f"{new_col}: {df[new_col].mean():.3f}")

In [ ]:
subset_fraction = 0.2
train_df = df[df['SEP28k-E']== 'train'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
valid_df = df[df['SEP28k-E']== 'dev'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
test_df = df[df['SEP28k-E']== 'test'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# load voc2vec Encoder
print("\n Loading voc2vec encoder...")
# MODEL_NAME = "alkiskoudounas/voc2vec-hubert-ls-pt" # facebook/wav2vec2-base, alkiskoudounas/voc2vec-hubert-ls-pt, openai/whisper-large-v3-turbo
MODEL_NAME = "facebook/wav2vec2-base"
# MODEL_NAME = "openai/whisper-large-v3-turbo"
encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")

In [ ]:
import soundfile as sf
import numpy as np

# Hyperparameters
batch_size    = 256
num_epochs    = 50
learning_rate = 0.0001

SAVE_PATH = '/content/drive/MyDrive/stutter_embeddings/'
os.makedirs(SAVE_PATH, exist_ok=True)

def filter_valid(df):
    valid = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Filtering"):
        try:
            if sf.info(row['filepath']).frames == 48000:
                valid.append(row)
        except:
            pass
    return pd.DataFrame(valid).reset_index(drop=True)

print("Filtering train...")
train_df_valid = filter_valid(train_df)
print("Filtering val...")
val_df_valid   = filter_valid(valid_df)
print("Filtering test...")
test_df_valid  = filter_valid(test_df)

print(f"Train valid: {len(train_df_valid)}")
print(f"Val valid:   {len(val_df_valid)}")
print(f"Test valid:  {len(test_df_valid)}")

In [ ]:
class StutterDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe):
        self.df         = dataframe.reset_index(drop=True)
        self.label_cols = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']
        self.labels     = torch.tensor(self.df[self.label_cols].values, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sample_rate = torchaudio.load(row['filepath'])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
        name = f"{row['Show']}_{row['EpId']}_{row['ClipId']}"
        return waveform, self.labels[idx], name

def make_raw_loader(df):
    return DataLoader(
        StutterDataset(df),
        batch_size=16,
        num_workers=0,
        pin_memory=False,
        shuffle=False
    )

train_loader_raw = make_raw_loader(train_df_valid)
val_loader_raw   = make_raw_loader(val_df_valid)
test_loader_raw  = make_raw_loader(test_df_valid)

print(f"Train batches: {len(train_loader_raw)}")
print(f"Val batches:   {len(val_loader_raw)}")
print(f"Test batches:  {len(test_loader_raw)}")

In [ ]:
def precompute_embeddings(loader, split_name):
    save_file = SAVE_PATH + f'{split_name}_embeddings.npy'
    if os.path.exists(save_file):
        print(f"✓ {split_name} already exists, skipping")
        return

    encoder.eval()
    emb_file   = open(SAVE_PATH + f'{split_name}_embeddings_tmp.bin', 'wb')
    label_file = open(SAVE_PATH + f'{split_name}_labels_tmp.bin',     'wb')
    names_list = []
    total      = 0

    with torch.no_grad():
        for waveforms, labels, names in tqdm(loader, desc=f"Extracting {split_name}"):
            waveforms = waveforms.squeeze(1).to(device)
            with torch.cuda.amp.autocast():
                emb = encoder(waveforms).last_hidden_state
            emb_file.write(emb.cpu().float().numpy().tobytes())
            label_file.write(labels.numpy().tobytes())
            names_list.extend(names)
            total += len(waveforms)
            del emb, waveforms
            torch.cuda.empty_cache()

    emb_file.close()
    label_file.close()

    print(f"Converting {split_name} to npy...")
    emb_data   = np.frombuffer(
        open(SAVE_PATH + f'{split_name}_embeddings_tmp.bin', 'rb').read(),
        dtype=np.float32
    ).reshape(total, 149, 768)
    label_data = np.frombuffer(
        open(SAVE_PATH + f'{split_name}_labels_tmp.bin', 'rb').read(),
        dtype=np.float32
    ).reshape(total, 5)

    np.save(SAVE_PATH + f'{split_name}_embeddings.npy', emb_data)
    np.save(SAVE_PATH + f'{split_name}_labels.npy',     label_data)
    np.save(SAVE_PATH + f'{split_name}_names.npy',      np.array(names_list))

    os.remove(SAVE_PATH + f'{split_name}_embeddings_tmp.bin')
    os.remove(SAVE_PATH + f'{split_name}_labels_tmp.bin')
    print(f"✓ Saved {split_name}: ({total}, 149, 768)")
    del emb_data, label_data
    gc.collect()

precompute_embeddings(train_loader_raw, 'train')

In [ ]:
precompute_embeddings(val_loader_raw, 'val')

In [ ]:
precompute_embeddings(test_loader_raw, 'test')

del encoder
torch.cuda.empty_cache()
gc.collect()
print("✓ Encoder freed")

In [ ]:
import shutil

# Copy from Drive to local Colab disk (much faster reads)
LOCAL_PATH = '/content/embeddings/'
os.makedirs(LOCAL_PATH, exist_ok=True)

print("Copying embeddings to local disk...")
for split in ['train', 'val', 'test']:
    for f in ['embeddings', 'labels', 'names']:
        src = SAVE_PATH + f'{split}_{f}.npy'
        dst = LOCAL_PATH + f'{split}_{f}.npy'
        if not os.path.exists(dst):
            print(f"  Copying {split}_{f}.npy...")
            shutil.copy(src, dst)
        else:
            print(f"  ✓ {split}_{f}.npy already local")
print("✓ All embeddings local")

In [ ]:
# Check available RAM
import psutil
ram = psutil.virtual_memory()
print(f"Available RAM: {ram.available/1e9:.1f} GB")
print(f"Total RAM:     {ram.total/1e9:.1f} GB")

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, split_name, in_ram=False):
        print(f"Loading {split_name}...")

        if in_ram:
            self.embeddings = torch.from_numpy(
                np.load(LOCAL_PATH + f'{split_name}_embeddings.npy')
            ).float()
            self.labels = torch.from_numpy(
                np.load(LOCAL_PATH + f'{split_name}_labels.npy')
            ).float()
            self.in_ram = True
        else:
            self.embeddings = np.load(LOCAL_PATH + f'{split_name}_embeddings.npy', mmap_mode='r')
            self.labels     = np.load(LOCAL_PATH + f'{split_name}_labels.npy',     mmap_mode='r')
            self.in_ram = False

        self.names = np.load(LOCAL_PATH + f'{split_name}_names.npy', mmap_mode='r')
        ram = psutil.virtual_memory()
        print(f"✓ {split_name}: {self.embeddings.shape} | RAM {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB ({ram.percent}%)")

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        if self.in_ram:
            return self.embeddings[idx], self.labels[idx], str(self.names[idx])
        else:
            return (
                torch.tensor(self.embeddings[idx].copy(), dtype=torch.float32),
                torch.tensor(self.labels[idx].copy(),     dtype=torch.float32),
                str(self.names[idx])
            )

def make_embedding_loader(split_name, shuffle=False, in_ram=False):
    return DataLoader(
        EmbeddingDataset(split_name, in_ram=in_ram),
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        shuffle=shuffle
    )

# Train + val in RAM, test as mmap
train_loader = make_embedding_loader('train', shuffle=True, in_ram=True)
val_loader   = make_embedding_loader('val',                 in_ram=True)
test_loader  = make_embedding_loader('test',                in_ram=False)  # mmap only

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
from torchaudio.models import Conformer
import torch
import torch.nn as nn

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, x):
        scores  = self.attention(x)               # (B, T, 1)
        weights = torch.softmax(scores, dim=1)    # (B, T, 1)
        pooled  = torch.sum(x * weights, dim=1)   # (B, hidden)
        return pooled


class MultiScaleConformer(nn.Module):
    def __init__(self, input_dim=768, num_heads=4, ffn_dim=512, num_layers=1, dropout=0.1):
        super().__init__()

        self.conformer_short = Conformer(
            input_dim=input_dim,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            num_layers=num_layers,
            depthwise_conv_kernel_size=15,
            dropout=dropout
        )

        self.conformer_medium = Conformer(
            input_dim=input_dim,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            num_layers=num_layers,
            depthwise_conv_kernel_size=31,
            dropout=dropout
        )

        self.conformer_long = Conformer(
            input_dim=input_dim,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            num_layers=num_layers,
            depthwise_conv_kernel_size=63,
            dropout=dropout
        )

    def forward(self, x):
        # x: (B, T, 768)
        lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long).to(x.device)

        out_short,  _ = self.conformer_short(x,  lengths)  # (B, T, 768)
        out_medium, _ = self.conformer_medium(x, lengths)  # (B, T, 768)
        out_long,   _ = self.conformer_long(x,   lengths)  # (B, T, 768)

        # Concat → (B, T, 2304)
        return torch.cat([out_short, out_medium, out_long], dim=-1)


class StutterNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Downsample 768 → 256 using Conv (not linear projection)
        self.downsample = nn.Conv1d(768, 256, kernel_size=1)

        self.conformer = MultiScaleConformer(
            input_dim=256,
            num_heads=4,
            ffn_dim=512,
            num_layers=1,
            dropout=0.1
        )

        self.attention = AttentionPooling(hidden_size=256 * 3)

        self.dense = nn.Sequential(
            nn.Linear(256 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.head_prolongation = nn.Linear(128, 1)
        self.head_block        = nn.Linear(128, 1)
        self.head_soundrep     = nn.Linear(128, 1)
        self.head_wordrep      = nn.Linear(128, 1)
        self.head_interjection = nn.Linear(128, 1)

    def forward(self, x):
        # x: (B, T, 768)

        # Conv1d expects (B, C, T) — permute first
        x = x.permute(0, 2, 1)          # (B, 768, T)
        x = self.downsample(x)           # (B, 256, T)
        x = x.permute(0, 2, 1)          # (B, T, 256)

        # Multi-Scale Conformer → (B, T, 768)
        x = self.conformer(x)

        # Attention Pooling → (B, 768)
        x = self.attention(x)

        # Dense → (B, 128)
        x = self.dense(x)

        return {
            'prolongation': self.head_prolongation(x).squeeze(-1),
            'block':        self.head_block(x).squeeze(-1),
            'soundrep':     self.head_soundrep(x).squeeze(-1),
            'wordrep':      self.head_wordrep(x).squeeze(-1),
            'interjection': self.head_interjection(x).squeeze(-1)
        }

model = StutterNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params/1e6:.1f}M")
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scaler    = GradScaler(device=device)

eval_accu   = []
eval_losses = []
classes     = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']


def train(epoch):
    model.train()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    for data in tqdm(train_loader, desc="Training"):
        inputs  = data[0].to(device, non_blocking=True)
        targets = {h: data[1][:, i].to(device, non_blocking=True)
                   for i, h in enumerate(classes)}

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = 0
            for h in classes:
                head_loss = criterion(outputs[h], targets[h])
                running_loss[h] += head_loss.item()
                loss += head_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for h in classes:
            predicted = torch.round(torch.sigmoid(outputs[h]))
            total[h]   += targets[h].size(0)
            correct[h] += predicted.eq(targets[h]).sum().item()

    for h in classes:
        writer.add_scalar(f"{h} Loss/train",     running_loss[h] / len(train_loader), epoch)
        writer.add_scalar(f"{h} Accuracy/train", 100. * correct[h] / total[h],        epoch)


def eval(epoch):
    model.eval()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    with torch.no_grad():
        for data in tqdm(val_loader, desc="Validating"):
            inputs  = data[0].to(device, non_blocking=True)
            targets = {h: data[1][:, i].to(device, non_blocking=True)
                       for i, h in enumerate(classes)}

            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                for h in classes:
                    head_loss = criterion(outputs[h], targets[h])
                    running_loss[h] += head_loss.item()

            for h in classes:
                predicted  = torch.round(torch.sigmoid(outputs[h]))
                total[h]   += targets[h].size(0)
                correct[h] += predicted.eq(targets[h]).sum().item()

    for h in classes:
        validate_loss = running_loss[h] / len(val_loader)
        accu          = 100. * correct[h] / total[h]
        writer.add_scalar(f"{h} Loss/eval",     validate_loss, epoch)
        writer.add_scalar(f"{h} Accuracy/eval", accu,          epoch)
        eval_accu.append(accu)
        eval_losses.append(validate_loss)

    return sum(running_loss[h] for h in classes) / (len(val_loader) * len(classes))

In [ ]:
from datetime import datetime
import os

best_val_loss    = float('inf')
train_start_time = datetime.now()

# Save to Drive so it persists if Colab disconnects
MODEL_SAVE_PATH = '/content/drive/MyDrive/stutter_embeddings/DisfluencyDetector_best.pth'

for epoch in tqdm(range(1, num_epochs + 1), desc="Epoch"):
    train(epoch)
    eval_loss = eval(epoch)

    if eval_loss < best_val_loss:
        best_val_loss = eval_loss
        torch.save({
            'epoch':      epoch,
            'model':      model.state_dict(),
            'optimizer':  optimizer.state_dict(),
            'val_loss':   best_val_loss,
        }, MODEL_SAVE_PATH)
        print(f"  ✓ Saved best model (val_loss: {best_val_loss:.4f})")

    torch.cuda.empty_cache()

train_end_time = datetime.now()
print(f"\nFinished training in {train_end_time - train_start_time}")
print(f"Best val loss: {best_val_loss:.4f}")

writer.close()

In [ ]:
model.eval()

classToOutputMap = {
    'block':        'Block',
    'soundrep':     'SoundRep',
    'wordrep':      'WordRep',
    'interjection': 'Interjection',
    'prolongation': 'Prolongation'
}

total             = {h: 0 for h in classes}
correct           = {h: 0 for h in classes}
running_loss      = {h: 0 for h in classes}
predicted_stutter = {h: 0 for h in classes}
labels_stutter    = {h: 0 for h in classes}
correct_stutter   = {h: 0 for h in classes}

exportColumns = {'Clip': []}
for h in classes:
    exportColumns[classToOutputMap[h]] = []

with torch.no_grad():
    for data in tqdm(test_loader, desc="Testing"):
        inputs     = data[0].to(device, non_blocking=True)
        clip_names = data[2]
        targets    = {h: data[1][:, i].to(device, non_blocking=True)
                      for i, h in enumerate(classes)}

        with torch.cuda.amp.autocast():
            outputs = model(inputs)

        exportColumns['Clip'] += list(clip_names)

        for h in classes:
            head_loss        = criterion(outputs[h], targets[h])
            running_loss[h] += head_loss.item()

            probs            = torch.sigmoid(outputs[h])
            predicted_labels = torch.round(probs)

            total[h]   += targets[h].size(0)
            correct[h] += predicted_labels.eq(targets[h]).sum().item()

            predicted_stutter[h] += predicted_labels.sum().item()
            labels_stutter[h]    += targets[h].sum().item()
            correct_stutter[h]   += (predicted_labels * targets[h]).sum().item()

            exportColumns[classToOutputMap[h]] += probs.tolist()

# Export to Drive
exportDf = pd.DataFrame(exportColumns)
exportDf.to_csv('/content/drive/MyDrive/stutter_embeddings/results.csv', index=False)
print("Saved results.csv to Drive")

# Metrics
print("\n" + "="*50)
for h in classes:
    accu      = 100. * correct[h] / total[h]
    test_loss = running_loss[h] / len(test_loader)
    precision = correct_stutter[h] / predicted_stutter[h] if predicted_stutter[h] > 0 else 0
    recall    = correct_stutter[h] / labels_stutter[h]    if labels_stutter[h]    > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n{h} {'─'*40}")
    print(f"  Accuracy  : {accu:.2f}%")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Test Loss : {test_loss:.4f}")

In [ ]:
exportColumns['Block']